In [14]:
import yfinance as yf
import pandas as pd
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

In [15]:
tickers = [
    "NVDA",
    "AAPL",
    "MSFT",
    "AMZN",
    "GOOGL",
    "AVGO",
    "META",
    "GOOG",
    "MU",
    "AMD"
]

historic_start = "2015-01-01"
historic_end = "2026-01-06"
stocks = yf.download(
    tickers,
    start=historic_start,
    end=historic_end ,
    auto_adjust=False
)

[*********************100%***********************]  10 of 10 completed


In [16]:
prices = stocks["Close"]
returns = prices.pct_change().dropna()

In [17]:
returns.tail()

Ticker,AAPL,AMD,AMZN,AVGO,GOOG,GOOGL,META,MSFT,MU,NVDA
Date,,,,,,,,,,
2025-12-29,0.001317,0.002884,-0.001935,-0.007781,-0.001810,0.000159,-0.006935,-0.001251,0.033639,-0.012124
2025-12-30,-0.002484,-0.001252,0.001982,0.001317,0.000509,0.000925,0.011022,0.000780,-0.005911,-0.003613
2025-12-31,-0.004468,-0.005480,-0.007354,-0.010719,-0.002384,-0.002708,-0.008799,-0.007918,-0.024673,-0.005545
2026-01-02,-0.003127,0.043472,-0.018716,0.004392,0.004844,0.006869,-0.014665,-0.022083,0.105147,0.012601
2026-01-05,-0.013837,-0.010695,0.028962,-0.012082,0.006343,0.004411,0.012884,-0.000190,-0.010367,-0.003866


In [18]:
def add_bank_asset(
    returns_df,
    bank_rate=0.05
):
    bank_daily_return = (1 + bank_rate) ** (1 / 252) - 1

    bank_returns = pd.Series(
        bank_daily_return,
        index=returns_df.index,
        name="Bank"
    )

    return pd.concat(
        [returns_df, bank_returns],
        axis=1
    )

def calculate_period_returns(
    daily_returns,
    month_ids,
    year_ids
):
    # --------------------------------------------------
    # Monthly
    # --------------------------------------------------

    monthly_log_returns = tf.math.unsorted_segment_sum(
        tf.math.log1p(daily_returns),
        month_ids,
        tf.reduce_max(month_ids) + 1
    )

    monthly_days = tf.math.unsorted_segment_sum(
        tf.ones_like(daily_returns),
        month_ids,
        tf.reduce_max(month_ids) + 1
    )

    monthly_daily_returns = (
        tf.exp(
            monthly_log_returns / monthly_days
        ) - 1
    )

    
    # --------------------------------------------------
    # Yearly
    # --------------------------------------------------

    yearly_log_returns = tf.math.unsorted_segment_sum(
        tf.math.log1p(daily_returns),
        year_ids,
        tf.reduce_max(year_ids) + 1
    )

    yearly_days = tf.math.unsorted_segment_sum(
        tf.ones_like(daily_returns),
        year_ids,
        tf.reduce_max(year_ids) + 1
    )

    yearly_daily_returns = (
        tf.exp(
            yearly_log_returns / yearly_days
        ) - 1
    )

    return (
        monthly_daily_returns,
        yearly_daily_returns
    )
def create_portfolio_inputs(
    returns_df,
    window
):
    stock_returns = returns_df.iloc[:, :10].to_numpy(
        dtype=np.float32
    )

    asset_returns = returns_df.to_numpy(
        dtype=np.float32
    )

    X = []
    y = []

    for i in range(window, len(returns_df)):

        X.append(
            stock_returns[i-window:i].reshape(-1)
        )

        y.append(
            asset_returns[i]
        )

    return (
        np.asarray(X, dtype=np.float32),
        np.asarray(y, dtype=np.float32),
        returns_df.index[window:]
    )


def split_train_validation(
    X,
    y,
    dates,
    validation_fraction=0.2
):
    split = int(
        len(X) * (1 - validation_fraction)
    )

    return (
        X[:split],
        y[:split],
        dates[:split],
        X[split:],
        y[split:],
        dates[split:]
    )



def create_period_ids(dates):
    dates = pd.DatetimeIndex(dates)

    month_ids, _ = pd.factorize(
        dates.to_period("M")
    )

    year_ids, _ = pd.factorize(
        dates.to_period("Y")
    )

    return (
        tf.constant(month_ids, dtype=tf.int32),
        tf.constant(year_ids, dtype=tf.int32)
    )

def create_week_ids(dates):

    dates = pd.DatetimeIndex(dates)

    week_ids, _ = pd.factorize(
        dates.to_period("W")
    )

    return tf.constant(
        week_ids,
        dtype=tf.int32
    )


In [19]:
class BoundedPortfolioWeights(layers.Layer):

    def __init__(
        self,
        min_weights,
        max_weights,
        **kwargs
    ):
        super().__init__(**kwargs)

        self.min_weights = tf.constant(
            min_weights,
            dtype=tf.float32
        )

        self.max_weights = tf.constant(
            max_weights,
            dtype=tf.float32
        )

        self.total_min = tf.reduce_sum(
            self.min_weights
        )

        self.remaining = (
            1.0 - self.total_min
        )

        self.capacities = (
            self.max_weights - self.min_weights
        )

        self.n_assets = len(min_weights)

    def call(self, logits):

        # --------------------------------------------------
        # Convert logits to positive scores
        # --------------------------------------------------

        scores = tf.exp(
            logits - tf.reduce_max(
                logits,
                axis=1,
                keepdims=True
            )
        )

        # --------------------------------------------------
        # Allocate the remaining portfolio weight
        # while respecting upper bounds
        # --------------------------------------------------

        q = scores / tf.reduce_sum(
            scores,
            axis=1,
            keepdims=True
        )

        fixed = tf.zeros_like(
            q,
            dtype=tf.bool
        )

        for _ in range(self.n_assets):

            # Current amount allocated to fixed assets
            fixed_allocation = tf.reduce_sum(
                tf.where(
                    fixed,
                    q,
                    tf.zeros_like(q)
                ),
                axis=1,
                keepdims=True
            )

            remaining_allocation = (
                1.0 - fixed_allocation
            )

            # Scores of unfixed assets
            free_scores = tf.where(
                fixed,
                tf.zeros_like(scores),
                scores
            )

            free_score_sum = tf.reduce_sum(
                free_scores,
                axis=1,
                keepdims=True
            )

            new_q = (
                remaining_allocation
                * free_scores
                / (
                    free_score_sum
                    + 1e-8
                )
            )

            # Keep already-fixed assets at their cap
            new_q = tf.where(
                fixed,
                q,
                new_q
            )

            # Maximum allowed allocation relative
            # to the remaining portfolio
            max_q = (
                self.capacities
                / self.remaining
            )

            # Identify assets that exceed their cap
            newly_fixed = (
                new_q > max_q
            )

            fixed = (
                fixed | newly_fixed
            )

            q = tf.where(
                newly_fixed,
                max_q,
                new_q
            )

        # --------------------------------------------------
        # Convert residual allocation back to weights
        # --------------------------------------------------

        weights = (
            self.min_weights
            + self.remaining * q
        )

        return weights

def build_strategy_1_model(
    window
):

    # 10 stocks
    stock_min = 1 / 121

    # Bank
    bank_min = 0.05

    min_weights = (
        [stock_min] * 10
        + [bank_min]
    )

    # Maximum for every asset
    max_weights = [0.20] * 11

    return keras.Sequential([
        layers.Input(shape=(window * 10,)),

        layers.Dense(
            32,
            activation="relu"
        ),

        layers.Dense(
            16,
            activation="relu"
        ),

        layers.Dense(
            11
        ),

        BoundedPortfolioWeights(
            min_weights=min_weights,
            max_weights=max_weights
        )
    ])


def train_one_episode(
    model,
    optimizer,
    X_train,
    y_train,
    month_ids,
    year_ids,
    week_ids,
    turnover_penalty=1.0,
    initial_weights=None,
    monthly_weight=0.25,
    yearly_weight=0.75
):

    with tf.GradientTape() as tape:

        # --------------------------------------------------
        # NN outputs
        # --------------------------------------------------

        daily_nn_weights = model(
            X_train,
            training=True
        )

        # --------------------------------------------------
        # Identify first trading day of each week
        # --------------------------------------------------

        week_starts = tf.concat(
            [
                tf.constant([True]),
                tf.not_equal(
                    week_ids[1:],
                    week_ids[:-1]
                )
            ],
            axis=0
        )

        # One NN portfolio per week
        weekly_nn_weights = tf.boolean_mask(
            daily_nn_weights,
            week_starts
        )

        # Number of trading days in each week
        week_counts = tf.math.unsorted_segment_sum(
            tf.ones_like(week_ids, dtype=tf.int32),
            week_ids,
            tf.reduce_max(week_ids) + 1
        )

        # --------------------------------------------------
        # Initial portfolio
        # --------------------------------------------------

        if initial_weights is None:

            initial_weights = tf.constant(
                [1 / 11] * 11,
                dtype=tf.float32
            )

        else:

            initial_weights = tf.convert_to_tensor(
                initial_weights,
                dtype=tf.float32
            )

        # First week uses initial weights
        # Subsequent weeks use NN weights
        weekly_weights = tf.concat(
            [
                initial_weights[tf.newaxis, :],
                weekly_nn_weights[1:]
            ],
            axis=0
        )

        # --------------------------------------------------
        # Hold the weekly weights constant
        # throughout each week
        # --------------------------------------------------

        weights = tf.repeat(
            weekly_weights,
            week_counts,
            axis=0
        )

        # --------------------------------------------------
        # Portfolio daily returns
        # --------------------------------------------------

        portfolio_daily_returns = tf.reduce_sum(
            weights * y_train,
            axis=1
        )

        monthly_returns, yearly_returns = (
            calculate_period_returns(
                portfolio_daily_returns,
                month_ids,
                year_ids
            )
        )

        # --------------------------------------------------
        # Return objective
        # --------------------------------------------------

        return_loss = -(
            monthly_weight * tf.reduce_mean(monthly_returns)
            + yearly_weight * tf.reduce_mean(yearly_returns)
        )

        # --------------------------------------------------
        # Weekly weight-change penalty
        # --------------------------------------------------

        weekly_weight_changes = (
            weekly_weights[1:]
            - weekly_weights[:-1]
        )

        turnover_penalty_value = tf.reduce_mean(
            tf.reduce_mean(
                tf.square(
                    weekly_weight_changes
                ),
                axis=1
            )
        )

        # --------------------------------------------------
        # Total loss
        # --------------------------------------------------

        loss = (
            return_loss
            + turnover_penalty * turnover_penalty_value
        )

    gradients = tape.gradient(
        loss,
        model.trainable_variables
    )

    optimizer.apply_gradients(
        zip(
            gradients,
            model.trainable_variables
        )
    )

    return (
        loss,
        portfolio_daily_returns,
        monthly_returns,
        yearly_returns,
        turnover_penalty_value
    )

def train_strategy_1(
    returns_df,
    window=20,
    validation_fraction=0.2,
    n_episodes=200,
    learning_rate=0.001,
    bank_rate=0.05,
    turnover_penalty=1.0,
    initial_weights=None,
    monthly_weight=0.25,
    yearly_weight=0.75
):

    # Add bank
    returns_with_bank = add_bank_asset(
        returns_df,
        bank_rate
    )

    # Create inputs
    X, y, dates = create_portfolio_inputs(
        returns_with_bank,
        window
    )

    # Train / validation split
    (
        X_train,
        y_train,
        train_dates,
        X_val,
        y_val,
        val_dates
    ) = split_train_validation(
        X,
        y,
        dates,
        validation_fraction
    )

    # Period IDs
    train_month_ids, train_year_ids = (
        create_period_ids(train_dates)
    )
    
    val_month_ids, val_year_ids = (
        create_period_ids(val_dates)
    )
    
    # Weekly IDs
    train_week_ids = create_week_ids(
        train_dates
    )
    
    val_week_ids = create_week_ids(
        val_dates
    )

    # Build model
    model = build_strategy_1_model(
        window
    )

    optimizer = keras.optimizers.Adam(
        learning_rate=learning_rate
    )

    # Training
    history = []

    for episode in range(n_episodes):

        (
            loss,
            daily_returns,
            monthly_returns,
            yearly_returns,
            turnover_penalty_value
        ) = train_one_episode(
            model,
            optimizer,
            X_train,
            y_train,
            train_month_ids,
            train_year_ids,
            train_week_ids,
            turnover_penalty=turnover_penalty,
            initial_weights=initial_weights,
            monthly_weight=monthly_weight,
            yearly_weight=yearly_weight
        )
    
        print(
            f"Episode {episode + 1}/{n_episodes} "
            f"- Loss: {loss.numpy():.6f} "
            f"- Avg Monthly Return: "
            f"{tf.reduce_mean(monthly_returns).numpy():.6f} "
            f"- Weight Penalty: "
            f"{turnover_penalty_value.numpy():.6f}"
        )
        history.append({
            "episode": episode + 1,
            "loss": float(loss.numpy()),
            "avg_monthly_return": float(
                tf.reduce_mean(monthly_returns).numpy()
            ),
            "avg_yearly_return": float(
                tf.reduce_mean(yearly_returns).numpy()
            )
        })

    history = pd.DataFrame(history)

    # Validation
    val_weights = model(
        X_val,
        training=False
    )

    val_daily_returns = tf.reduce_sum(
        val_weights * y_val,
        axis=1
    )

    val_monthly_returns, val_yearly_returns = (
        calculate_period_returns(
            val_daily_returns,
            val_month_ids,
            val_year_ids
        )
    )

    return {
        "model": model,
        "history": history,
        "X_train": X_train,
        "y_train": y_train,
        "X_val": X_val,
        "y_val": y_val,
        "validation_dates": val_dates,
        "validation_daily_returns":
            val_daily_returns.numpy(),
        "validation_monthly_returns":
            val_monthly_returns.numpy(),
        "validation_yearly_returns":
            val_yearly_returns.numpy()
    }

In [20]:
# ==========================================================
# Prepare evaluation returns
# ==========================================================

def prepare_evaluation_returns(
    returns_df,
    bank_rate=0.05
):
    
    stock_returns = returns_df.iloc[:, :10].to_numpy(
        dtype=np.float32
    )

    bank_daily_return = (
        (1 + bank_rate) ** (1 / 252) - 1
    )

    bank_returns = np.full(
        (len(stock_returns), 1),
        bank_daily_return,
        dtype=np.float32
    )

    asset_returns = np.concatenate(
        [stock_returns, bank_returns],
        axis=1
    )

    return asset_returns


# ==========================================================
# Create evaluation inputs
# ==========================================================

def create_evaluation_inputs(
    returns_df,
    window
):
    
    stock_returns = returns_df.iloc[:, :10].to_numpy(
        dtype=np.float32
    )

    X = []

    for i in range(window, len(returns_df)):

        X.append(
            stock_returns[i-window:i].reshape(-1)
        )

    X = np.asarray(
        X,
        dtype=np.float32
    )

    evaluation_dates = pd.DatetimeIndex(
        returns_df.index[window:]
    )

    return X, evaluation_dates


# ==========================================================
# Validate initial portfolio weights
# ==========================================================

def validate_initial_weights(
    initial_weights=None
):
    
    stock_min = 1 / 121
    bank_min = 0.05

    min_weights = np.array(
        [stock_min] * 10 + [bank_min],
        dtype=np.float32
    )

    max_weights = np.array(
        [0.20] * 11,
        dtype=np.float32
    )

    if initial_weights is None:

        initial_weights = np.array(
            [1 / 11] * 11,
            dtype=np.float32
        )

    else:

        initial_weights = np.asarray(
            initial_weights,
            dtype=np.float32
        )

        if len(initial_weights) != 11:
            raise ValueError(
                "initial_weights must contain exactly 11 weights."
            )

        if not np.isclose(
            initial_weights.sum(),
            1.0
        ):
            raise ValueError(
                "initial_weights must sum to 1."
            )

        if np.any(
            initial_weights < min_weights
        ):
            raise ValueError(
                "initial_weights violate the minimum "
                "weight constraints."
            )

        if np.any(
            initial_weights > max_weights
        ):
            raise ValueError(
                "initial_weights violate the maximum "
                "weight constraint."
            )

    return initial_weights


# ==========================================================
# Create weekly portfolio weights
# ==========================================================

def create_weekly_evaluation_weights(
    nn_weights,
    evaluation_dates,
    initial_weights
):
    
    evaluation_weeks = (
        pd.DatetimeIndex(evaluation_dates)
        .to_period("W")
    )

    # First trading day of each week
    week_starts = np.r_[
        0,
        1 + np.flatnonzero(
            evaluation_weeks[1:]
            != evaluation_weeks[:-1]
        )
    ]

    # Number of trading days in each week
    week_counts = np.diff(
        np.r_[
            week_starts,
            len(evaluation_dates)
        ]
    )

    # NN prediction on first trading day
    # of each week
    weekly_nn_weights = (
        nn_weights[week_starts]
    )

    # First week uses initial portfolio.
    # Subsequent weeks use NN portfolio.
    weekly_weights = np.vstack([
        initial_weights,
        weekly_nn_weights[1:]
    ])

    # Hold each weekly portfolio constant
    # throughout that week
    evaluation_weights = np.repeat(
        weekly_weights,
        week_counts,
        axis=0
    )

    return evaluation_weights


# ==========================================================
# Calculate evaluation metrics
# ==========================================================

def calculate_evaluation_metrics(
    evaluation_dates,
    daily_returns
):

    results = pd.DataFrame({
        "date": evaluation_dates,
        "NN_daily_return": daily_returns
    })

    # --------------------------------------------------
    # Monthly returns
    # --------------------------------------------------

    results["month"] = (
        results["date"].dt.to_period("M")
    )

    monthly = results.groupby("month").agg({
        "NN_daily_return":
            lambda x: np.prod(1 + x) - 1
    }).reset_index()

    # --------------------------------------------------
    # Yearly returns
    # --------------------------------------------------

    results["year"] = (
        results["date"].dt.to_period("Y")
    )

    yearly = results.groupby("year").agg({
        "NN_daily_return":
            lambda x: np.prod(1 + x) - 1
    }).reset_index()

    # --------------------------------------------------
    # Summary
    # --------------------------------------------------

    summary = {
        "NN_average_daily_return":
            results["NN_daily_return"].mean(),

        "NN_average_monthly_return":
            monthly["NN_daily_return"].mean(),

        "NN_average_yearly_return":
            yearly["NN_daily_return"].mean()
    }

    return {
        "daily": results,
        "monthly": monthly,
        "yearly": yearly,
        "summary": summary
    }



In [21]:
# ==========================================================
# Train Strategy 1
# ==========================================================

def train_strategy_1(
    returns_df,
    window=20,
    n_episodes=200,
    learning_rate=0.001,
    bank_rate=0.05,
    turnover_penalty=1.0,
    initial_weights=None,
    monthly_weight=0.25,
    yearly_weight=0.75
):
    
    # --------------------------------------------------
    # Add bank asset
    # --------------------------------------------------
    
    stock_returns = returns_df.iloc[:, :10].copy()
    
    bank_daily_return = (
        (1 + bank_rate) ** (1 / 252) - 1
    )
    
    bank_returns = pd.Series(
        bank_daily_return,
        index=stock_returns.index,
        name="Bank"
    )
    
    asset_returns = stock_returns.copy()
    asset_returns["Bank"] = bank_returns
    
    
    # --------------------------------------------------
    # Create model inputs
    # --------------------------------------------------
    
    X, y, dates = create_portfolio_inputs(
        asset_returns,
        window
    )
    
    
    # --------------------------------------------------
    # Create period IDs
    # --------------------------------------------------
    
    month_ids, year_ids = create_period_ids(
        dates
    )
    
    week_ids = create_week_ids(
        dates
    )
    
    
    # --------------------------------------------------
    # Build model
    # --------------------------------------------------
    
    model = build_strategy_1_model(
        window
    )
    
    
    # --------------------------------------------------
    # Optimizer
    # --------------------------------------------------
    
    optimizer = keras.optimizers.Adam(
        learning_rate=learning_rate
    )
    
    
    # --------------------------------------------------
    # Convert inputs to tensors
    # --------------------------------------------------
    
    X_train = tf.convert_to_tensor(
        X,
        dtype=tf.float32
    )
    
    y_train = tf.convert_to_tensor(
        y,
        dtype=tf.float32
    )
    
    
    # --------------------------------------------------
    # Train
    # --------------------------------------------------
    
    history = []
    
    for episode in range(n_episodes):
        
        (
            loss,
            portfolio_daily_returns,
            monthly_returns,
            yearly_returns,
            turnover_penalty_value
        ) = train_one_episode(
            model=model,
            optimizer=optimizer,
            X_train=X_train,
            y_train=y_train,
            month_ids=month_ids,
            year_ids=year_ids,
            week_ids=week_ids,
            turnover_penalty=turnover_penalty,
            initial_weights=initial_weights,
            monthly_weight=monthly_weight,
            yearly_weight=yearly_weight
        )
        
        history.append({
            "episode": episode + 1,
            "loss": float(loss.numpy()),
            "turnover_penalty":
                float(turnover_penalty_value.numpy())
        })
    
    
    # --------------------------------------------------
    # Return trained model
    # --------------------------------------------------
    
    return {
        "model": model,
        "history": history,
        "dates": dates
    }

In [22]:
# ==========================================================
# Evaluate Strategy 1
# ==========================================================

def evaluate_strategy_1(
    model,
    returns_df,
    window=20,
    bank_rate=0.05,
    initial_weights=None
):
    
    # --------------------------------------------------
    # Prepare returns
    # --------------------------------------------------
    
    asset_returns = prepare_evaluation_returns(
        returns_df,
        bank_rate=bank_rate
    )
    
    
    # --------------------------------------------------
    # Create model inputs
    # --------------------------------------------------
    
    X, evaluation_dates = create_evaluation_inputs(
        returns_df,
        window
    )
    
    
    # --------------------------------------------------
    # NN portfolio weights
    # --------------------------------------------------
    
    nn_weights = model(
        X,
        training=False
    ).numpy()
    
    
    # --------------------------------------------------
    # Validate initial weights
    # --------------------------------------------------
    
    initial_weights = validate_initial_weights(
        initial_weights
    )
    
    
    # --------------------------------------------------
    # Create weekly portfolio weights
    # --------------------------------------------------
    
    evaluation_weights = create_weekly_evaluation_weights(
        nn_weights,
        evaluation_dates,
        initial_weights
    )
    
    
    # --------------------------------------------------
    # Future asset returns
    # --------------------------------------------------
    
    future_returns = asset_returns[window:]
    
    
    # --------------------------------------------------
    # Portfolio daily returns
    # --------------------------------------------------
    
    nn_daily_returns = np.sum(
        evaluation_weights * future_returns,
        axis=1
    )
    
    
    # --------------------------------------------------
    # Calculate performance metrics
    # --------------------------------------------------
    
    metrics = calculate_evaluation_metrics(
        evaluation_dates,
        nn_daily_returns
    )
    
    
    # --------------------------------------------------
    # Store weights
    # --------------------------------------------------
    
    metrics["weights"] = evaluation_weights
    
    
    return metrics

In [23]:
# ==========================================================
# Prepare starting weights
# ==========================================================

def prepare_initial_weights(
    initial_weights=None
):
    
    if initial_weights is None:
        return np.array(
            [1 / 11] * 11,
            dtype=np.float32
        )
    
    return np.asarray(
        initial_weights,
        dtype=np.float32
    )
# ==========================================================
# Calculate risk metrics from daily returns
# ==========================================================

def calculate_risk_metrics(
    daily_returns,
    confidence_level=0.95
):
    
    daily_returns = np.asarray(
        daily_returns,
        dtype=np.float64
    )
    
    # --------------------------------------------------
    # VaR
    # 95% confidence -> 5% quantile
    # --------------------------------------------------
    
    var = np.percentile(
        daily_returns,
        (1 - confidence_level) * 100
    )
    
    
    # --------------------------------------------------
    # CVaR
    # Expected return conditional on being
    # below the 5% quantile
    # --------------------------------------------------
    
    tail_returns = daily_returns[
        daily_returns <= var
    ]
    
    cvar = tail_returns.mean()
    
    
    return {
        "VaR": var,
        "CVaR": cvar
    }


# ==========================================================
# Calculate benchmark returns
# ==========================================================
# ==========================================================
# Calculate benchmark returns
# ==========================================================

def calculate_benchmarks(
    returns_df,
    bank_rate=0.05
):
    
    # --------------------------------------------------
    # Stock returns
    # --------------------------------------------------
    
    stock_returns = returns_df.iloc[:, :10].to_numpy(
        dtype=np.float32
    )
    
    
    # --------------------------------------------------
    # Bank returns
    # --------------------------------------------------
    
    bank_daily_return = (
        (1 + bank_rate) ** (1 / 252) - 1
    )
    
    bank_returns = np.full(
        (len(returns_df), 1),
        bank_daily_return,
        dtype=np.float32
    )
    
    
    # --------------------------------------------------
    # Combine stocks and bank
    # --------------------------------------------------
    
    asset_returns = np.concatenate(
        [stock_returns, bank_returns],
        axis=1
    )
    
    
    # ==================================================
    # Benchmark 1:
    # Stocks + 5% Bank
    # ==================================================
    
    stocks_bank_weights = np.array(
        [0.095] * 10 + [0.05],
        dtype=np.float32
    )
    
    stocks_bank_daily_returns = np.sum(
        asset_returns * stocks_bank_weights,
        axis=1
    )
    
    stocks_bank_return = (
        np.prod(
            1 + stocks_bank_daily_returns
        ) - 1
    )
    
    
    # ==================================================
    # Benchmark 2:
    # Stocks Only
    # ==================================================
    
    stocks_only_weights = np.array(
        [1 / 10] * 10,
        dtype=np.float32
    )
    
    stocks_only_daily_returns = np.sum(
        stock_returns * stocks_only_weights,
        axis=1
    )
    
    stocks_only_return = (
        np.prod(
            1 + stocks_only_daily_returns
        ) - 1
    )
    
    
    # ==================================================
    # Benchmark 3:
    # Fully Equal Stocks + Bank
    # ==================================================
    
    fully_equal_weights = np.array(
        [1 / 11] * 11,
        dtype=np.float32
    )
    
    fully_equal_daily_returns = np.sum(
        asset_returns * fully_equal_weights,
        axis=1
    )
    
    fully_equal_return = (
        np.prod(
            1 + fully_equal_daily_returns
        ) - 1
    )
    
    
    return {
        # --------------------------------------------------
        # Full returns
        # --------------------------------------------------
        
        "Stocks + 5% Bank":
            stocks_bank_return,
        
        "Stocks Only":
            stocks_only_return,
        
        "Fully Equal Stocks + Bank":
            fully_equal_return,
        
        
        # --------------------------------------------------
        # Daily returns
        # --------------------------------------------------
        
        "Stocks + 5% Bank Daily Returns":
            stocks_bank_daily_returns,
        
        "Stocks Only Daily Returns":
            stocks_only_daily_returns,
        
        "Fully Equal Stocks + Bank Daily Returns":
            fully_equal_daily_returns
    }



# ==========================================================
# Calculate final results table
# ==========================================================

def calculate_results_table(
    evaluations,
    benchmarks
):
    
    results = []
    
    
    # ======================================================
    # NN strategies
    # ======================================================
    
    for window, evaluation in evaluations.items():
        
        nn_daily_returns = (
            evaluation["daily"]["NN_daily_return"]
            .to_numpy()
        )
        
        # Full return
        nn_full_return = (
            np.prod(
                1 + nn_daily_returns
            ) - 1
        )
        
        # Risk
        nn_risk = calculate_risk_metrics(
            nn_daily_returns
        )
        
        results.append({
            "Strategy": f"Window Size {window}",
            "Full Return": nn_full_return * 100,
            "VaR 95%": nn_risk["VaR"] * 100,
            "CVaR 95%": nn_risk["CVaR"] * 100
        })
    
    
    # ======================================================
    # Stocks Only
    # ======================================================
    
    stocks_only_risk = calculate_risk_metrics(
        benchmarks["Stocks Only Daily Returns"]
    )
    
    results.append({
        "Strategy": "Stocks Only",
        "Full Return":
            benchmarks["Stocks Only"] * 100,
        "VaR 95%":
            stocks_only_risk["VaR"] * 100,
        "CVaR 95%":
            stocks_only_risk["CVaR"] * 100
    })
    
    
    # ======================================================
    # Stocks + 5% Bank
    # ======================================================
    
    stocks_bank_risk = calculate_risk_metrics(
        benchmarks["Stocks + 5% Bank Daily Returns"]
    )
    
    results.append({
        "Strategy": "Stocks + 5% Bank",
        "Full Return":
            benchmarks["Stocks + 5% Bank"] * 100,
        "VaR 95%":
            stocks_bank_risk["VaR"] * 100,
        "CVaR 95%":
            stocks_bank_risk["CVaR"] * 100
    })
    
    
    # ======================================================
    # Fully Equal Stocks + Bank
    # ======================================================
    
    fully_equal_risk = calculate_risk_metrics(
        benchmarks["Fully Equal Stocks + Bank Daily Returns"]
    )
    
    results.append({
        "Strategy": "Fully Equal Stocks + Bank",
        "Full Return":
            benchmarks["Fully Equal Stocks + Bank"] * 100,
        "VaR 95%":
            fully_equal_risk["VaR"] * 100,
        "CVaR 95%":
            fully_equal_risk["CVaR"] * 100
    })
    
    
    return pd.DataFrame(results)


# ==========================================================
# Evaluate all strategies
# ==========================================================

def evaluate_all_strategies(
    strategies,
    test_returns,
    initial_weights
):
    
    evaluations = {}
    
    for window, strategy in strategies.items():
        
        evaluations[window] = evaluate_strategy_1(
            strategy["model"],
            test_returns,
            window=window,
            initial_weights=initial_weights
        )
    
    return evaluations

# ==========================================================
# Main experiment function
# ==========================================================

# ==========================================================
# Train and evaluate all strategies
# ==========================================================
# ==========================================================
# Train all strategies
# ==========================================================

def train_all_strategies(
    returns,
    turnover_penalty,
    initial_weights,
    monthly_weight,
    yearly_weight
):
    
    windows = [5, 10, 20]
    
    strategies = {}
    
    for window in windows:
        
        strategies[window] = train_strategy_1(
            returns,
            window=window,
            turnover_penalty=turnover_penalty,
            initial_weights=initial_weights,
            monthly_weight=monthly_weight,
            yearly_weight=yearly_weight
        )
    
    return strategies

# ==========================================================
# Train and evaluate all strategies
# ==========================================================

def run_strategy_evaluation(
    train_returns,
    evaluation_returns,
    turnover_penalties=[5, 1, 0.5],
    initial_weights=None,
    bank_rate=0.05,
    monthly_weight=0.25,
    yearly_weight=0.75
):
    
    # --------------------------------------------------
    # Starting portfolio
    # --------------------------------------------------
    
    initial_weights = prepare_initial_weights(
        initial_weights
    )
    
    
    # --------------------------------------------------
    # Store results
    # --------------------------------------------------
    
    all_strategies = {}
    all_evaluations = {}
    results = []
    
    
    # ==================================================
    # Train NN strategies
    # ==================================================
    
    for turnover_penalty in turnover_penalties:
        
        # --------------------------------------------------
        # Train all 3 window sizes
        # --------------------------------------------------
        
        strategies = train_all_strategies(
            train_returns,
            turnover_penalty,
            initial_weights,
            monthly_weight,
            yearly_weight
        )
        
        all_strategies[turnover_penalty] = strategies
        
        
        # --------------------------------------------------
        # Evaluate each window
        # --------------------------------------------------
        
        evaluations = {}
        
        for window, strategy in strategies.items():
            
            # Use the last `window` training observations
            # to construct the first evaluation input.
            evaluation_data = pd.concat([
                train_returns.iloc[-window:],
                evaluation_returns
            ])
            
            evaluation = evaluate_strategy_1(
                strategy["model"],
                evaluation_data,
                window=window,
                bank_rate=bank_rate,
                initial_weights=initial_weights
            )
            
            evaluations[window] = evaluation
            
            
            # --------------------------------------------------
            # Evaluation daily returns
            # --------------------------------------------------
            
            daily_returns = (
                evaluation["daily"]["NN_daily_return"]
                .to_numpy()
            )
            
            
            # --------------------------------------------------
            # Full evaluation return
            # --------------------------------------------------
            
            full_return = (
                np.prod(
                    1 + daily_returns
                ) - 1
            )
            
            
            # --------------------------------------------------
            # Risk
            # --------------------------------------------------
            
            risk = calculate_risk_metrics(
                daily_returns
            )
            
            
            # --------------------------------------------------
            # Store NN result
            # --------------------------------------------------
            
            results.append({
                "Strategy":
                    f"Window Size {window}",
                
                "Turnover Penalty":
                    turnover_penalty,
                
                "Evaluation Return":
                    full_return * 100,
                
                "VaR 95%":
                    risk["VaR"] * 100,
                
                "CVaR 95%":
                    risk["CVaR"] * 100
            })
        
        
        all_evaluations[turnover_penalty] = evaluations
    
    
    # ==================================================
    # Calculate benchmarks
    # ==================================================
    
    benchmarks = calculate_benchmarks(
        evaluation_returns,
        bank_rate=bank_rate
    )
    
    
    # ==================================================
    # Add benchmark results
    # ==================================================
    
    benchmark_names = [
        "Stocks Only",
        "Stocks + 5% Bank",
        "Fully Equal Stocks + Bank"
    ]
    
    for benchmark_name in benchmark_names:
    
        daily_returns = benchmarks[
            f"{benchmark_name} Daily Returns"
        ]
    
        full_return = benchmarks[
            benchmark_name
        ]
    
        risk = calculate_risk_metrics(
            daily_returns
        )
    
        results.append({
            "Strategy":
                benchmark_name,
    
            # Benchmarks do not use a turnover penalty
            "Turnover Penalty":
                0,
    
            "Evaluation Return":
                full_return * 100,
    
            "VaR 95%":
                risk["VaR"] * 100,
    
            "CVaR 95%":
                risk["CVaR"] * 100
        })
    
    
    # --------------------------------------------------
    # Results table
    # --------------------------------------------------
    
    evaluation_results = pd.DataFrame(
        results
    )
    
    
    # --------------------------------------------------
    # Display
    # --------------------------------------------------
    
    display(evaluation_results)
    
    
    # --------------------------------------------------
    # Return everything
    # --------------------------------------------------
    
    return {
        "results":
            evaluation_results,
        
        "strategies":
            all_strategies,
        
        "evaluations":
            all_evaluations,
        
        "benchmarks":
            benchmarks
    }

In [24]:
train_end = int(len(returns) * 0.60)
validation_end = int(len(returns) * 0.80)

train_returns = returns.iloc[:train_end]

validation_returns = returns.iloc[
    train_end:validation_end
]

In [25]:
validation_phase = run_strategy_evaluation(
    train_returns,
    validation_returns,
    turnover_penalties=[5, 1, 0.5]
)

,Strategy,Turnover Penalty,Evaluation Return,VaR 95%,CVaR 95%
0,Window Size 5,5.0,23.576450,-3.359348,-4.640174
1,Window Size 10,5.0,25.456417,-3.226816,-4.422109
2,Window Size 20,5.0,28.380526,-3.277560,-4.512218
3,Window Size 5,1.0,20.123636,-3.354201,-4.683604
4,Window Size 10,1.0,19.588614,-3.372014,-4.643450
5,Window Size 20,1.0,26.781023,-3.247938,-4.545066
6,Window Size 5,0.5,20.434547,-3.424947,-4.705937
7,Window Size 10,0.5,21.375942,-3.344648,-4.599880
8,Window Size 20,0.5,23.014904,-3.248039,-4.559622
9,Stocks Only,0.0,19.028664,-3.291514,-4.426432


In [ ]:

train_returns_80 = returns.iloc[:validation_end]

test_returns = returns.iloc[validation_end:]

# ==========================================================
# Phase 2: Train on 80%, Test on final 20%
# ==========================================================

test_phase = run_strategy_evaluation(
    train_returns_80,
    test_returns,
    turnover_penalties=[5, 1, 0.5]
)